# Fast Prompting en Acción — POC
**Proyecto:** Generador rápido de *briefs* de pantallas UX a partir de requisitos en texto.  
**Autor:** Marta Olivares  
**Curso:** Inteligencia Artificial: Generación de Prompts (Diplomatura)  
**Fecha:** 2025-09-12
---


## 1. Introducción
**Problema:** Cada vez que iniciamos una pantalla nueva en un proyecto de diseño UX/UI, se pierde tiempo transformando requisitos dispersos (mensajes, notas, tickets) en un *brief* claro y accionable para diseñar.  
**Relevancia:** Reducir ese tiempo de preparación acelera la entrega y baja costes. Un *brief* consistente también mejora la comunicación con stakeholders.

**Propuesta:** Una notebook que, mediante técnicas de **Fast Prompting**, convierta requisitos en un *brief* UX listo para trabajar, con estructura estándar y validada automáticamente.


## 2. Objetivos
1. Demostrar dominio de principios/técnicas de **Fast Prompting** con LLMs.
2. Experimentar configuraciones de prompts para optimizar calidad, coste y tiempo.
3. Entregar una **POC** funcional en Jupyter Notebook con resultados reproducibles.
4. Analizar mejoras respecto a la Preentrega 1 y justificar elecciones.


## 3. Metodología (resumen)
- **Diseño del output primero:** definimos un esquema JSON del *brief* (título, usuario objetivo, problema, objetivos, KPIs, requisitos de UI, restricciones, criterios de aceptación).
- **Fast Prompting** probado:
  - *CRISPE* (Context, Role, Intent, Style, Persona, Examples) / *CReST*.
  - **Instrucciones atómicas** y ordenadas + **esquema rígido (JSON)**.
  - **Few-shot** con 1–2 ejemplos sintéticos.
  - **Draft → Critique → Fix** (auto‑revisión barata con prompts cortos).
  - **Guardrails**: validación JSON y corrección automática.
- **Evaluación ligera**: métricas heurísticas (validez JSON, cobertura de campos, longitud razonable) y coste estimado por tokens.


## 4. Requisitos previos
- Python 3.10+
- Paquetes: `openai>=1.40`, `python-dotenv`, `pydantic`, `tiktoken` *(opcional)*, `pandas`.
- Variable de entorno `OPENAI_API_KEY` configurada.

> ⚠️ **Coste**: esta POC está optimizada para pocas llamadas. Por entrada hacemos 1–3 llamadas (según validez del JSON). Usa modelos económicos si lo prefieres.


In [ ]:
# !pip install -q openai>=1.40 python-dotenv pydantic tiktoken pandas

In [ ]:
from __future__ import annotations
import os, json
from typing import Dict, Any, List, Tuple

try:
    from openai import OpenAI
except Exception as e:
    print("Instala el SDK oficial de OpenAI: pip install openai>=1.40")
    OpenAI = None

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")  # económico y competente
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY) if (API_KEY and OpenAI) else None

def ensure_client():
    if client is None:
        raise RuntimeError("Configura OPENAI_API_KEY y reinstala 'openai' si es necesario.")


## 5. Esquema del *brief* UX (salida deseada)
Usamos un JSON estructurado para facilitar validación y reuso.


In [ ]:
BRIEF_SCHEMA = {
    "type": "object",
    "properties": {
        "pantalla": {"type": "string"},
        "usuario_objetivo": {"type": "string"},
        "problema": {"type": "string"},
        "objetivos": {"type": "array", "items": {"type": "string"}},
        "kpis": {"type": "array", "items": {"type": "string"}},
        "requisitos_ui": {"type": "array", "items": {"type": "string"}},
        "restricciones": {"type": "array", "items": {"type": "string"}},
        "criterios_aceptacion": {"type": "array", "items": {"type": "string"}},
        "notas": {"type": "string"}
    },
    "required": ["pantalla","usuario_objetivo","problema","objetivos","requisitos_ui","criterios_aceptacion"]
}

def is_valid_brief(obj: Dict[str, Any]) -> Tuple[bool, List[str]]:
    errors = []
    for key in BRIEF_SCHEMA["required"]:
        if key not in obj:
            errors.append(f"Falta campo requerido: {key}")
    if "objetivos" in obj and not isinstance(obj["objetivos"], list):
        errors.append("objetivos debe ser lista")
    if "requisitos_ui" in obj and not isinstance(obj["requisitos_ui"], list):
        errors.append("requisitos_ui debe ser lista")
    if "criterios_aceptacion" in obj and not isinstance(obj["criterios_aceptacion"], list):
        errors.append("criterios_aceptacion debe ser lista")
    return (len(errors) == 0, errors)


## 6. Datos de ejemplo (entrada)
Requisitos en texto tal y como llegarían por ticket/whatsapp/email.


In [ ]:
ENTRADAS_EJEMPLO = [
    {
        "id": "TCK-001",
        "texto": "Necesitamos una pantalla de registro súper simple para la app de cursos. Usuarios: mayores de 18 años, España. Solo email + contraseña + checkbox Términos. Debe incluir CTA claro y enlace a '¿Ya tienes cuenta? Inicia sesión'. Cumplir RGPD."
    },
    {
        "id": "TCK-002",
        "texto": "Dashboard inicial para conductores VTC en modo nocturno. KPI: ganancias por hora, viajes completados y cancelaciones. Acceso rápido a 'Aceptar nuevo viaje' y 'Pausar'. Diseño accesible, alto contraste. Debe funcionar bien con una mano."
    }
]

## 7. Configuraciones de Fast Prompting
Probamos 3 variantes crecientes en calidad/coste:
1. **Baseline**: instrucciones breves + esquema JSON.
2. **CRISPE + Few-shot**: rol claro, contexto, ejemplos mínimos.
3. **Draft→Critique→Fix**: si el JSON no valida, hacemos una pasada corta de corrección.


In [ ]:
BASE_SYSTEM = (
    "Eres un asistente experto en UX/UI que genera BRIEFS de pantallas en español de España. "
    "Responde exclusivamente con JSON válido conforme al esquema dado."
)

BASE_USER_TEMPLATE = "Convierte el siguiente requisito en un JSON de brief UX.\n" \
                      f"Esquema (claves exactas): {list(BRIEF_SCHEMA['properties'].keys())}\n"                       "Texto: \"{texto}\""

FEWSHOT_EXAMPLES = [
    {
        "input": "Pantalla de login para ecommerce. Email, password, recordar, recuperar password, CTA 'Entrar', enlace a registro. Cumplir RGPD.",
        "output": {
            "pantalla": "Login",
            "usuario_objetivo": "Compradores recurrentes del ecommerce",
            "problema": "Acceso rápido y seguro a su cuenta",
            "objetivos": ["Permitir inicio de sesión en <15s", "Reducir errores de acceso"],
            "kpis": ["Tasa de login exitoso", "Errores de password"],
            "requisitos_ui": [
                "Campos email y contraseña",
                "Checkbox 'Recordarme'",
                "Enlace '¿Olvidaste tu contraseña?'",
                "CTA principal 'Entrar'",
                "Enlace a 'Crear cuenta'"
            ],
            "restricciones": ["RGPD", "Accesibilidad AA"],
            "criterios_aceptacion": [
                "Validación de email en cliente",
                "Mensajes de error claros",
                "Navegable por teclado"
            ],
            "notas": "Usar enfoque mobile‑first"
        }
    }
]


In [ ]:
def chat(messages, model: str = MODEL, max_tokens: int = 500):
    ensure_client()
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.2,
        max_tokens=max_tokens,
        response_format={"type": "json_object"}
    )
    return resp.choices[0].message.content, resp.usage


In [ ]:
def run_baseline(texto: str):
    msgs = [
        {"role":"system", "content": BASE_SYSTEM},
        {"role":"user", "content": BASE_USER_TEMPLATE.format(texto=texto)}
    ]
    out, usage = chat(msgs)
    return out, usage

def run_crispe_fewshot(texto: str):
    msgs = [{"role":"system","content": BASE_SYSTEM + " Sigue CRISPE y usa el esquema estrictamente."}]
    for ex in FEWSHOT_EXAMPLES:
        msgs.append({"role":"user","content": BASE_USER_TEMPLATE.format(texto=ex["input"])})
        import json as _json
        msgs.append({"role":"assistant","content": _json.dumps(ex["output"], ensure_ascii=False)})
    msgs.append({"role":"user","content": BASE_USER_TEMPLATE.format(texto=texto)})
    out, usage = chat(msgs)
    return out, usage

def fix_if_invalid(json_text: str):
    try:
        data = json.loads(json_text)
        ok, errs = is_valid_brief(data)
        if ok:
            return data, {"total_tokens": 0, "prompt_tokens": 0, "completion_tokens": 0}
    except Exception:
        errs = ["JSON inválido/parseable"]
        data = None
    
    msgs = [
        {"role":"system", "content": "Arregla un JSON para que sea válido y cumpla un esquema dado. Responde solo JSON."},
        {"role":"user", "content": json.dumps({"schema_keys": list(BRIEF_SCHEMA["properties"].keys()), "json": json_text, "errores": errs}, ensure_ascii=False)}
    ]
    out, usage = chat(msgs, max_tokens=300)
    try:
        fixed = json.loads(out)
        return fixed, usage
    except Exception:
        return {"error": "No se pudo corregir automáticamente", "original": json_text}, usage


## 8. Experimentos
Ejecutamos las dos variantes principales y, si hace falta, una **pasada de corrección**. Registramos uso de tokens para estimar coste.


In [ ]:
import pandas as pd

def experimentar(entradas):
    registros = []
    resultados = {}
    for e in entradas:
        fila_id = e["id"]
        # Baseline
        out_b, u_b = run_baseline(e["texto"])
        fixed_b, u_fix_b = fix_if_invalid(out_b)
        ok_b, _ = is_valid_brief(fixed_b) if isinstance(fixed_b, dict) else (False, [])
        registros.append({
            "id": fila_id, "variant": "baseline", "ok": ok_b,
            "prompt_tokens": getattr(u_b, "prompt_tokens", None),
            "completion_tokens": getattr(u_b, "completion_tokens", None),
            "total_tokens": getattr(u_b, "total_tokens", None),
            "fix_tokens": getattr(u_fix_b, "total_tokens", None) if isinstance(u_fix_b, dict) else None
        })
        resultados[(fila_id, "baseline")] = fixed_b

        # CRISPE + few-shot
        out_c, u_c = run_crispe_fewshot(e["texto"])
        fixed_c, u_fix_c = fix_if_invalid(out_c)
        ok_c, _ = is_valid_brief(fixed_c) if isinstance(fixed_c, dict) else (False, [])
        registros.append({
            "id": fila_id, "variant": "crispe_fewshot", "ok": ok_c,
            "prompt_tokens": getattr(u_c, "prompt_tokens", None),
            "completion_tokens": getattr(u_c, "completion_tokens", None),
            "total_tokens": getattr(u_c, "total_tokens", None),
            "fix_tokens": getattr(u_fix_c, "total_tokens", None) if isinstance(u_fix_c, dict) else None
        })
        resultados[(fila_id, "crispe_fewshot")] = fixed_c

    df = pd.DataFrame(registros)
    return df, resultados

# Para correr los experimentos, descomenta:
# df_res, outs = experimentar(ENTRADAS_EJEMPLO)
# df_res


## 9. Selección de prompt final
Se escoge la variante que **valida JSON a la primera** con **menos tokens**. Debajo, función `brief_ux()` que expone la solución final.


In [ ]:
def brief_ux(texto: str, variant: str = "crispe_fewshot"):
    if variant == "baseline":
        raw, _ = run_baseline(texto)
    else:
        raw, _ = run_crispe_fewshot(texto)
    fixed, _ = fix_if_invalid(raw)
    if isinstance(fixed, dict):
        return fixed
    raise ValueError("No se pudo producir un JSON válido.")


In [ ]:
# Prueba rápida (ejecuta tras configurar OPENAI_API_KEY):
# import json as _json
# resultado = brief_ux("Pantalla de registro para app de cursos con email+password+TOS y enlace a login. RGPD.")
# print(_json.dumps(resultado, ensure_ascii=False, indent=2))


## 10. Coste estimado
La notebook registra `usage` de la API. Con una tabla de precios por 1K tokens puedes estimar el gasto. Ajusta con el modelo elegido.


In [ ]:
PRECIOS_KTOK = {
    # Actualiza estos valores según el modelo real que uses.
    "gpt-4o-mini": {"input": 0.00015, "output": 0.0006}  # USD por 1K tokens (ejemplo)
}

def estimar_coste(modelo: str, prompt_toks: int, completion_toks: int) -> float:
    p = PRECIOS_KTOK.get(modelo, {"input":0.0, "output":0.0})
    return (prompt_toks/1000.0)*p["input"] + (completion_toks/1000.0)*p["output"]


## 11. Demostración final
Completa el texto con tus requisitos y obtén el *brief* UX listo para usar.


In [ ]:
demo_texto = """
Necesitamos una pantalla para reportar un problema en una app de movilidad.
Debe tener selector de tipo de incidencia, campo de descripción, adjuntar foto,
y un botón 'Enviar'. Acceso a historial de reportes. Soporte modo oscuro.
""".strip()

# Descomenta para probar
# import json as _json
# print(_json.dumps(brief_ux(demo_texto), ensure_ascii=False, indent=2))


## 12. Conclusiones y mejoras
- **Mejora vs. Preentrega 1:** salida estandarizada en JSON, auto‑revisión y *few-shot* reducen trabajo manual y re‑intentos.
- **Coste:** 1–2 llamadas por caso en la mayoría de entradas. Se prioriza un modelo económico.
- **Futuro:** añadir interfaz ligera (Gradio/Streamlit), logging de coste real y *caching* de prompts.
